In [ ]:
from neo4j import GraphDatabase

class Neo4jApp:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()
        print("✅ Connected to Neo4j successfully!")

    def close(self):
        self.driver.close()
        print("🔌 Connection closed.")

    def insert_person(self, name, age):
        query = """
        MERGE (p:Person {name: $name})
        SET p.age = $age
        RETURN p
        """
        with self.driver.session() as session:
            result = session.run(query, name=name, age=age)
            record = result.single()
            print(f"Inserted/Updated Person: {record['p']['name']}")

    def create_friendship(self, person1_name, person2_name):
        query = """
        MATCH (a:Person {name: $person1_name})
        MATCH (b:Person {name: $person2_name})
        MERGE (a)-[r:KNOWS]->(b)
        RETURN r
        """
        with self.driver.session() as session:
            session.run(query, person1_name=person1_name, person2_name=person2_name)
            print(f"Created relationship: {person1_name} KNOWS {person2_name}")

    def insert_bulk_data(self, people_list):
        query = """
        UNWIND $batch AS person
        MERGE (p:Person {name: person.name})
        SET p.age = person.age
        """
        with self.driver.session() as session:
            session.run(query, batch=people_list)
            print(f"Bulk inserted {len(people_list)} people.")

In [ ]:
# Database Credentials
URI = "neo4j://localhost:7687"
USER = "neo4j"
PASSWORD = "your_secure_password" # Change this!

# Initialize the app
app = Neo4jApp(URI, USER, PASSWORD)

In [ ]:
app.insert_person("Alice", 30)
app.insert_person("Bob", 32)